# 7.2 — Classical Features

Classical vision features replace brittle raw-pixel matching with explicit measurements of local structure: gradients, edges, corners, orientation histograms, geometric votes, and small amounts of pooling. In this lesson, you will build those pieces from scratch with NumPy so the math behind SIFT, HOG, Hough lines, and hand-made filters is visible before later lessons learn similar filters automatically.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build classical features one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so no edge, histogram, vote, or descriptor is a black box. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, finite differences, histograms, and linear algebra.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for any random-looking toy images.

### 1. Finite differences turn pixels into gradients

A grayscale image is just a grid of intensities, so the first classical feature asks a local question: how much does intensity change from left to right and top to bottom? We approximate those changes with finite differences. If $G_x$ is the horizontal change and $G_y$ is the vertical change, then magnitude and orientation are

$$m=\sqrt{G_x^2+G_y^2},\qquad \theta=\operatorname{atan2}(G_y,G_x).$$

In [ ]:
img_w = np.array([[0., 0., 1., 1., 1.],
                  [0., 0., 1., 1., 1.],
                  [0., 0., 1., 1., 1.],
                  [0., 0., 1., 1., 1.],
                  [0., 0., 1., 1., 1.]])  # a vertical step edge.
Gx_w = np.zeros_like(img_w)  # same shape as the image.
Gy_w = np.zeros_like(img_w)  # same shape as the image.
Gx_w[:, 1:-1] = (img_w[:, 2:] - img_w[:, :-2]) / 2  # centered left-right difference.
Gy_w[1:-1, :] = (img_w[2:, :] - img_w[:-2, :]) / 2  # centered top-bottom difference.
print("Gx middle row:", Gx_w[2])
print("Gy middle row:", Gy_w[2])
assert np.allclose(Gx_w[2], [0., 0.5, 0.5, 0., 0.])

▶ What you'll see: horizontal gradients appear on the two columns straddling the step; vertical gradients stay zero because the image does not change top-to-bottom.

In [ ]:
mag_w = np.sqrt(Gx_w**2 + Gy_w**2)  # gradient strength at each pixel.
ang_w = np.degrees(np.arctan2(Gy_w, Gx_w))  # gradient direction in degrees for readability.
print("magnitude middle row:", mag_w[2])
print("angle middle row:", ang_w[2])
assert round(float(mag_w[2, 1]), 3) == 0.5

▶ What you'll see: the edge pixels have magnitude 0.5 and angle 0°, meaning intensity rises in the +x direction.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6))
ax[0].imshow(img_w, cmap="gray", vmin=0, vmax=1); ax[0].set_title("image")
ax[1].imshow(Gx_w, cmap="coolwarm", vmin=-.5, vmax=.5); ax[1].set_title("Gx")
ax[2].imshow(mag_w, cmap="magma", vmin=0, vmax=.6); ax[2].set_title("magnitude")
for a in ax: a.axis("off")
plt.suptitle("1: finite differences expose the step"); plt.show()

▶ What you'll see: the raw step becomes a bright vertical band in the gradient-magnitude image.

*Why it's done this way:* a pixel value alone is tied to lighting, but a difference compares neighboring pixels and cancels much of the absolute brightness. The magnitude formula is the Euclidean length of the 2-D change vector, so horizontal and vertical evidence combine into one edge-strength number while orientation keeps the direction information for later descriptors.

### 2. Hand-designed filters slide local tests into feature maps

A filter is a deliberately chosen comparison. The diagonal kernel below says "top-left bright, bottom-right dark" because it multiplies one corner by +1 and the opposite corner by −1. Sliding the same test over every valid window turns one local measurement into a feature map.

In [ ]:
patch_w = np.array([[1., 2.], [3., 4.]])
kernel_w = np.array([[1., 0.], [0., -1.]])
products_w = patch_w * kernel_w
response_w = float(products_w.sum())
print("element products:\n", products_w)
print("filter response:", response_w)
assert response_w == -3.0

▶ What you'll see: only the top-left and bottom-right pixels matter, giving `1 - 4 = -3`.

In [ ]:
ramp_w = np.arange(1, 10, dtype=float).reshape(3, 3)
feat_w = np.zeros((2, 2))
for i_w in range(2):
    for j_w in range(2):
        feat_w[i_w, j_w] = np.sum(ramp_w[i_w:i_w+2, j_w:j_w+2] * kernel_w)
print("ramp:\n", ramp_w)
print("feature map:\n", feat_w)
assert np.all(feat_w == -4.0)

▶ What you'll see: every 2×2 window has the same diagonal gap, so the entire feature map is −4.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.6, 2.6))
ax[0].imshow(ramp_w, cmap="viridis"); ax[0].set_title("input ramp")
ax[1].imshow(feat_w, cmap="coolwarm", vmin=-5, vmax=5); ax[1].set_title("filter responses")
for a in ax: a.axis("off")
plt.suptitle("2: one local test, slid everywhere"); plt.show()

▶ What you'll see: the feature map is smaller than the input because only full 2×2 windows are evaluated.

*Why it's done this way:* classical filters encode the structure we care about as weights before learning enters the story. Multiplication selects and signs each pixel's contribution, summation collapses the window into one score, and sliding shares the same detector across positions so a pattern is recognized wherever it appears.

### 3. HOG summarizes gradients with orientation histograms

Histogram of Oriented Gradients (HOG) keeps gradient directions and magnitudes but forgets exact pixel identities inside a small cell. Each gradient vector votes into an orientation bin with weight equal to its magnitude, so strong edges count more than weak edges.

In [ ]:
gradients_w = np.array([[3., 4.], [0., 2.], [-3., 4.], [4., 0.]])
gx_w, gy_w = gradients_w[:, 0], gradients_w[:, 1]
mag_hog_w = np.sqrt(gx_w**2 + gy_w**2)
ang_hog_w = (np.degrees(np.arctan2(gy_w, gx_w)) + 180) % 180  # unsigned orientations.
print("magnitudes:", mag_hog_w)
print("angles:", np.round(ang_hog_w, 1))
assert np.allclose(mag_hog_w, [5., 2., 5., 4.])

▶ What you'll see: the four vectors have magnitudes 5, 2, 5, and 4 with unsigned angles near 53°, 90°, 127°, and 0°.

In [ ]:
bins_w = np.array([0., 90., 180.])
hist_w = np.zeros(2)
for m_w, a_w in zip(mag_hog_w, ang_hog_w):
    bin_id_w = 0 if a_w < 45 or a_w >= 135 else 1
    hist_w[bin_id_w] += m_w
print("two-bin HOG histogram:", hist_w)
assert np.allclose(hist_w, [4., 12.])

▶ What you'll see: the upright-ish bin receives 12 units of edge strength and the horizontal bin receives 4.

In [ ]:
plt.figure(figsize=(4.4, 3))
plt.bar(["0° bin", "90° bin"], hist_w, color=["steelblue", "darkorange"])
plt.title("3: HOG cell orientation votes")
plt.ylabel("sum of gradient magnitudes")
plt.show()

▶ What you'll see: one tall bar says the cell is dominated by vertical/upright structure.

*Why it's done this way:* a single gradient can be noisy or shift by a pixel, but a histogram pools many local gradients into a stable summary. Weighting by magnitude makes confident edges dominate, and dropping exact locations buys small-translation robustness at the cost of precise texture detail.

### 4. Corners are places where gradients change in two directions

An edge changes strongly in one direction but slides freely along the other; a corner changes in both directions. The Harris-style second-moment matrix collects local sums of $G_x^2$, $G_y^2$, and $G_xG_y$. Large energy in both axes means a small window cannot move horizontally or vertically without changing its appearance.

In [ ]:
corner_img_w = np.zeros((7, 7))
corner_img_w[3:, 3:] = 1.0  # a bright quadrant creates an L-shaped corner at (3,3).
Gx_c_w = np.zeros_like(corner_img_w); Gy_c_w = np.zeros_like(corner_img_w)
Gx_c_w[:, 1:-1] = (corner_img_w[:, 2:] - corner_img_w[:, :-2]) / 2
Gy_c_w[1:-1, :] = (corner_img_w[2:, :] - corner_img_w[:-2, :]) / 2
patch_slice_w = (slice(2, 5), slice(2, 5))
Sxx_w = float(np.sum(Gx_c_w[patch_slice_w] ** 2))
Syy_w = float(np.sum(Gy_c_w[patch_slice_w] ** 2))
Sxy_w = float(np.sum(Gx_c_w[patch_slice_w] * Gy_c_w[patch_slice_w]))
M_w = np.array([[Sxx_w, Sxy_w], [Sxy_w, Syy_w]])
print("second-moment matrix:\n", M_w)
assert np.allclose(np.diag(M_w), [1.0, 1.0])

▶ What you'll see: both diagonal entries are positive, showing horizontal and vertical gradient energy in the same neighborhood.

In [ ]:
eigs_w = np.linalg.eigvalsh(M_w)
R_corner_w = np.linalg.det(M_w) - 0.04 * (np.trace(M_w) ** 2)
print("eigenvalues:", np.round(eigs_w, 3))
print("corner response:", round(float(R_corner_w), 3))
assert round(float(R_corner_w), 3) == 0.777

▶ What you'll see: both eigenvalues are nonzero, so the response is positive instead of edge-like.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6))
ax[0].imshow(corner_img_w, cmap="gray"); ax[0].set_title("corner image")
ax[1].imshow(np.abs(Gx_c_w), cmap="magma"); ax[1].set_title("|Gx|")
ax[2].imshow(np.abs(Gy_c_w), cmap="magma"); ax[2].set_title("|Gy|")
for a in ax: a.axis("off")
plt.suptitle("4: a corner has two-direction gradient energy"); plt.show()

▶ What you'll see: the vertical edge lights up in `|Gx|`, the horizontal edge lights up in `|Gy|`, and their neighborhood overlap marks the corner.

*Why it's done this way:* the second-moment matrix measures how quickly a patch changes under tiny shifts. One large eigenvalue means an edge because motion along the edge barely changes the patch; two large eigenvalues mean a corner because every small motion changes the patch, making it useful for matching.

### 5. SIFT-style descriptors normalize orientation before describing a patch

SIFT's famous trick is not just detecting gradients; it builds a local coordinate frame. First estimate the dominant orientation of a keypoint neighborhood, rotate all local gradient angles relative to that dominant direction, then histogram the relative angles. This buys rotation robustness because the descriptor follows the patch.

In [ ]:
angles_sift_w = np.array([10., 20., 30., 190., 200., 210.])
mags_sift_w = np.array([2., 3., 4., 2., 3., 4.])
bin_edges_sift_w = np.arange(0, 361, 45)
hist_abs_w, _ = np.histogram(angles_sift_w, bins=bin_edges_sift_w, weights=mags_sift_w)
dom_bin_w = int(np.argmax(hist_abs_w))
dom_angle_w = 22.5 + 45 * dom_bin_w
print("absolute orientation histogram:", hist_abs_w)
print("dominant orientation:", dom_angle_w)
assert dom_angle_w == 22.5

▶ What you'll see: the first 45° bin wins, so the patch chooses a dominant orientation near 22.5°.

In [ ]:
rotated_angles_w = (angles_sift_w - dom_angle_w) % 360
hist_rel_w, _ = np.histogram(rotated_angles_w, bins=bin_edges_sift_w, weights=mags_sift_w)
print("relative angles:", np.round(rotated_angles_w, 1))
print("relative histogram:", hist_rel_w)
assert hist_rel_w.sum() == mags_sift_w.sum()

▶ What you'll see: the same gradient energy is now expressed relative to the keypoint's own orientation.

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(np.arange(len(hist_abs_w)) - 0.18, hist_abs_w, width=.36, label="absolute")
plt.bar(np.arange(len(hist_rel_w)) + 0.18, hist_rel_w, width=.36, label="relative")
plt.title("5: SIFT-style orientation normalization")
plt.xlabel("45° orientation bin")
plt.ylabel("weighted votes")
plt.legend(); plt.show()

▶ What you'll see: the histogram shifts after normalization, but total gradient evidence is preserved.

*Why it's done this way:* matching raw angles would fail when the object rotates. Subtracting the dominant orientation changes the descriptor from image coordinates to keypoint coordinates, deliberately discarding absolute pose so patches with the same internal structure can match after rotation.

### 6. Hough voting finds a line from many weak edge pixels

The Hough transform lets each edge pixel vote for geometric explanations. For lines, one convenient form is $\rho=x\cos\theta+y\sin\theta$. If several points lie on the same vertical line $x=2$, then at $\theta=0^\circ$ they all vote for $\rho=2$.

In [ ]:
points_w = np.array([[2., 0.], [2., 1.], [2., 3.]])
thetas_deg_w = np.array([0., 45., 90.])
rhos_w = []
for th_w in np.radians(thetas_deg_w):
    rhos_w.append(points_w[:, 0] * np.cos(th_w) + points_w[:, 1] * np.sin(th_w))
rhos_w = np.array(rhos_w)
print("rho votes by theta rows:\n", np.round(rhos_w, 3))
assert np.allclose(rhos_w[0], [2., 2., 2.])

▶ What you'll see: only the 0° row gives the same rho for all three points.

In [ ]:
rho_bins_w = np.arange(0, 6)
acc_w = np.zeros((len(thetas_deg_w), len(rho_bins_w)))
for t_w in range(len(thetas_deg_w)):
    for rho_w in rhos_w[t_w]:
        nearest_w = int(np.argmin(np.abs(rho_bins_w - rho_w)))
        acc_w[t_w, nearest_w] += 1
best_w = np.unravel_index(np.argmax(acc_w), acc_w.shape)
print("accumulator:\n", acc_w.astype(int))
print("best theta, rho:", thetas_deg_w[best_w[0]], rho_bins_w[best_w[1]])
assert acc_w[0, 2] == 3

▶ What you'll see: the accumulator peak is three votes at `(theta=0°, rho=2)`.

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(acc_w, cmap="magma", aspect="auto")
plt.xticks(range(len(rho_bins_w)), rho_bins_w); plt.yticks(range(len(thetas_deg_w)), thetas_deg_w)
plt.xlabel("rho bin"); plt.ylabel("theta (degrees)")
plt.title("6: Hough accumulator peak")
plt.colorbar(label="votes"); plt.show()

▶ What you'll see: one bright cell turns three separate edge pixels into one detected vertical line.

*Why it's done this way:* a single edge point is ambiguous because infinitely many lines pass through it. Voting changes the question from "which line does this pixel prove?" to "which line do many pixels agree on?"; the accumulator peak is robust because unrelated noise rarely lands in the same parameter bin.

### 7. Pooling and nonmaximum suppression trade detail for stability

Classical pipelines often keep the strongest local evidence and suppress nearby duplicates. Max pooling forgets the exact within-block position of a response, while nonmaximum suppression keeps only local peaks so one object or corner does not produce a cluster of redundant detections.

In [ ]:
act_w = np.array([[1., 3., 2., 0.],
                  [4., 6., 5., 1.],
                  [1., 2., 9., 8.],
                  [0., 1., 7., 3.]])
pooled_w = np.zeros((2, 2))
for i_w in range(2):
    for j_w in range(2):
        block_w = act_w[2*i_w:2*i_w+2, 2*j_w:2*j_w+2]
        pooled_w[i_w, j_w] = np.max(block_w)
print("pooled map:\n", pooled_w)
assert np.allclose(pooled_w, [[6., 5.], [2., 9.]])

▶ What you'll see: each 2×2 block keeps only its strongest activation: 6, 5, 2, and 9.

In [ ]:
scores_w = np.array([0.1, 0.8, 0.7, 0.2, 0.9, 0.85, 0.1])
keep_w = np.zeros_like(scores_w, dtype=bool)
for idx_w in range(1, len(scores_w) - 1):
    keep_w[idx_w] = scores_w[idx_w] >= scores_w[idx_w-1] and scores_w[idx_w] >= scores_w[idx_w+1]
print("scores:", scores_w)
print("local maxima kept at:", np.where(keep_w)[0])
assert np.array_equal(np.where(keep_w)[0], [1, 4])

▶ What you'll see: neighboring high scores collapse to peaks at positions 1 and 4.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7, 2.8))
ax[0].imshow(act_w, cmap="viridis"); ax[0].set_title("activation map")
ax[1].plot(scores_w, marker="o", label="scores")
ax[1].scatter(np.where(keep_w)[0], scores_w[keep_w], color="red", label="kept peaks")
ax[1].set_title("1-D nonmaximum suppression"); ax[1].legend()
plt.suptitle("7: keep strong evidence, drop local detail"); plt.show()

▶ What you'll see: pooling compresses blocks, and suppression keeps isolated peaks instead of dense duplicate responses.

*Why it's done this way:* invariance is purchased by discarding information. Pooling says a strong response somewhere nearby is enough; nonmaximum suppression says one local winner should represent a neighborhood. Both make downstream decisions less jumpy, but both can hurt precise localization when the discarded position matters.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each classical-feature mechanic by hand.** Separate from the walkthrough
> above, here is one tiny, fully hand-traceable toy per computational mechanic in this lesson. Each
> toy prints intermediates with real `# ->` values, draws one picture, and checks the result.

### ✍️ Toy 1 · Finite differences turn intensity changes into gradients

Centered differences compare left/right and top/bottom neighbors. A vertical step produces horizontal
gradients but no vertical gradients.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)                 # seeded generator for this toy
t1_img = np.array([[0., 0., 1., 1., 1.],
                   [0., 0., 1., 1., 1.],
                   [0., 0., 1., 1., 1.],
                   [0., 0., 1., 1., 1.],
                   [0., 0., 1., 1., 1.]])         # vertical step edge
print("middle image row:", t1_img[2].tolist())   # -> [0.0, 0.0, 1.0, 1.0, 1.0]
t1_Gx = np.zeros_like(t1_img)                    # -> 5x5 zeros
t1_Gy = np.zeros_like(t1_img)                    # -> 5x5 zeros
t1_Gx[:, 1:-1] = (t1_img[:, 2:] - t1_img[:, :-2]) / 2 # centered x-difference
t1_Gy[1:-1, :] = (t1_img[2:, :] - t1_img[:-2, :]) / 2 # centered y-difference
print("Gx middle row:", t1_Gx[2].tolist())       # -> [0.0, 0.5, 0.5, 0.0, 0.0]
print("Gy middle row:", t1_Gy[2].tolist())       # -> [0.0, 0.0, 0.0, 0.0, 0.0]
t1_mag = np.sqrt(t1_Gx**2 + t1_Gy**2)            # edge strength
print("magnitude row:", t1_mag[2].tolist())      # -> [0.0, 0.5, 0.5, 0.0, 0.0]
t1_ang = np.degrees(np.arctan2(t1_Gy, t1_Gx))    # -> direction in degrees
print("angle row:", t1_ang[2].tolist())          # -> [0.0, 0.0, 0.0, 0.0, 0.0]
assert np.allclose(t1_Gx[2], [0., 0.5, 0.5, 0., 0.]) and np.allclose(t1_Gy, 0.0)

fig, t1_ax = plt.subplots(1, 3, figsize=(7.4, 2.4))
t1_ax[0].imshow(t1_img, cmap="gray", vmin=0, vmax=1)
t1_ax[0].set_title("image")
t1_ax[1].imshow(t1_Gx, cmap="coolwarm", vmin=-0.5, vmax=0.5)
t1_ax[1].set_title("Gx")
t1_ax[2].imshow(t1_mag, cmap="magma", vmin=0, vmax=0.6)
t1_ax[2].set_title("magnitude")
for t1_a in t1_ax:
    t1_a.axis("off")
plt.suptitle("Toy 1 · finite differences expose the step")
plt.show()

▶ What you'll see: two bright gradient columns appear exactly around the vertical step.

### ✍️ Toy 2 · A hand-designed filter slides into a feature map

A small kernel scores one patch by multiply-and-sum, then the same local test slides over every valid
window.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)                 # seeded generator for this toy
t2_ramp = np.arange(1, 10, dtype=float).reshape(3, 3) # -> [[1,2,3],[4,5,6],[7,8,9]]
print("ramp:", t2_ramp.tolist())                 # -> [[1.0, 2.0, 3.0], [4.0, 5.0, 6.0], [7.0, 8.0, 9.0]]
t2_kernel = np.array([[1., 0.], [0., -1.]])      # -> diagonal difference
print("kernel:", t2_kernel.tolist())             # -> [[1.0, 0.0], [0.0, -1.0]]
t2_patch = t2_ramp[:2, :2]                       # -> [[1.0, 2.0], [4.0, 5.0]]
print("first patch:", t2_patch.tolist())         # -> [[1.0, 2.0], [4.0, 5.0]]
t2_products = t2_patch * t2_kernel               # -> [[1.0, 0.0], [0.0, -5.0]]
print("products:", t2_products.tolist())         # -> [[1.0, 0.0], [0.0, -5.0]]
t2_response = float(t2_products.sum())           # -> -4.0
print("first response:", t2_response)            # -> -4.0
t2_feature = np.zeros((2, 2))                    # output feature map
for t2_i in range(2):
    for t2_j in range(2):
        t2_feature[t2_i, t2_j] = np.sum(t2_ramp[t2_i:t2_i+2, t2_j:t2_j+2] * t2_kernel)
print("feature map:", t2_feature.tolist())       # -> [[-4.0, -4.0], [-4.0, -4.0]]
assert np.allclose(t2_feature, -4.0) and t2_response == -4.0

fig, t2_ax = plt.subplots(1, 2, figsize=(5.2, 2.5))
t2_ax[0].imshow(t2_ramp, cmap="viridis")
t2_ax[0].set_title("input ramp")
t2_ax[1].imshow(t2_feature, cmap="coolwarm", vmin=-5, vmax=5)
t2_ax[1].set_title("filter map")
for t2_a in t2_ax:
    t2_a.axis("off")
plt.suptitle("Toy 2 · one filter, many landings")
plt.show()

▶ What you'll see: every 2×2 window has the same diagonal gap, so every response is `-4`.

### ✍️ Toy 3 · HOG bins gradients by orientation with magnitude votes

HOG keeps direction but pools exact positions. Each gradient adds its magnitude to an orientation bin.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)                 # seeded generator for this toy
t3_gradients = np.array([[2., 0.], [0., 3.], [-4., 0.], [0., -1.]]) # -> four (Gx,Gy) vectors
print("gradients:", t3_gradients.tolist())       # -> [[2.0, 0.0], [0.0, 3.0], [-4.0, 0.0], [0.0, -1.0]]
t3_gx = t3_gradients[:, 0]                       # -> [2.0, 0.0, -4.0, 0.0]
t3_gy = t3_gradients[:, 1]                       # -> [0.0, 3.0, 0.0, -1.0]
t3_mag = np.sqrt(t3_gx**2 + t3_gy**2)            # -> [2.0, 3.0, 4.0, 1.0]
print("magnitudes:", t3_mag.tolist())            # -> [2.0, 3.0, 4.0, 1.0]
t3_ang = (np.degrees(np.arctan2(t3_gy, t3_gx)) + 180) % 180 # -> unsigned degrees
print("unsigned angles:", t3_ang.tolist())       # -> [0.0, 90.0, 0.0, 90.0]
t3_hist = np.zeros(2)                            # -> [0-degree bin, 90-degree bin]
for t3_m, t3_a in zip(t3_mag, t3_ang):
    t3_bin = 0 if t3_a < 45 or t3_a >= 135 else 1
    t3_hist[t3_bin] += t3_m
print("two-bin HOG:", t3_hist.tolist())          # -> [6.0, 4.0]
assert np.array_equal(t3_hist, np.array([6., 4.]))

plt.figure(figsize=(4.2, 2.7))
plt.bar(["0° bin", "90° bin"], t3_hist, color=["steelblue", "darkorange"])
plt.ylabel("sum of magnitudes")
plt.title("Toy 3 · HOG orientation votes")
plt.show()

▶ What you'll see: horizontal evidence totals `6`, while vertical evidence totals `4`.

### ✍️ Toy 4 · Harris-style corners need gradient energy in two directions

The second-moment matrix sums `Gx²`, `Gy²`, and `GxGy` in a local patch. A corner has two nonzero
eigenvalues.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)                 # seeded generator for this toy
t4_img = np.zeros((5, 5))                         # dark canvas
t4_img[2:, 2:] = 1.0                             # bright bottom-right quadrant
print("corner image:", t4_img.tolist())          # -> [[0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 1.0, 1.0, 1.0], [0.0, 0.0, 1.0, 1.0, 1.0], [0.0, 0.0, 1.0, 1.0, 1.0]]
t4_Gx = np.zeros_like(t4_img)                    # -> 5x5 zeros
t4_Gy = np.zeros_like(t4_img)                    # -> 5x5 zeros
t4_Gx[:, 1:-1] = (t4_img[:, 2:] - t4_img[:, :-2]) / 2 # x gradients
t4_Gy[1:-1, :] = (t4_img[2:, :] - t4_img[:-2, :]) / 2 # y gradients
print("Gx:", np.round(t4_Gx, 2).tolist())        # vertical-edge energy near column 2
print("Gy:", np.round(t4_Gy, 2).tolist())        # horizontal-edge energy near row 2
t4_patch = (slice(1, 4), slice(1, 4))             # -> 3x3 window around the corner
t4_Sxx = float(np.sum(t4_Gx[t4_patch] ** 2))      # -> 1.0
t4_Syy = float(np.sum(t4_Gy[t4_patch] ** 2))      # -> 1.0
t4_Sxy = float(np.sum(t4_Gx[t4_patch] * t4_Gy[t4_patch])) # -> 0.25
print("Sxx,Syy,Sxy:", t4_Sxx, t4_Syy, t4_Sxy)   # -> 1.0 1.0 0.25
t4_M = np.array([[t4_Sxx, t4_Sxy], [t4_Sxy, t4_Syy]]) # -> [[1.0,0.25],[0.25,1.0]]
print("second-moment matrix:", t4_M.tolist())    # -> [[1.0, 0.25], [0.25, 1.0]]
t4_eigs = np.linalg.eigvalsh(t4_M)               # -> [0.75, 1.25]
print("eigenvalues:", np.round(t4_eigs, 3).tolist()) # -> [0.75, 1.25]
t4_R = float(np.linalg.det(t4_M) - 0.04 * np.trace(t4_M) ** 2) # -> 0.7775
print("corner response:", round(t4_R, 3))        # -> 0.777
assert np.allclose(t4_eigs, [0.75, 1.25]) and round(t4_R, 3) == 0.777

fig, t4_ax = plt.subplots(1, 3, figsize=(7.2, 2.4))
t4_ax[0].imshow(t4_img, cmap="gray")
t4_ax[0].set_title("image")
t4_ax[1].imshow(np.abs(t4_Gx), cmap="magma")
t4_ax[1].set_title("|Gx|")
t4_ax[2].imshow(np.abs(t4_Gy), cmap="magma")
t4_ax[2].set_title("|Gy|")
for t4_a in t4_ax:
    t4_a.axis("off")
plt.suptitle("Toy 4 · corner = two directions")
plt.show()

▶ What you'll see: horizontal and vertical gradient energy overlap around the corner.

### ✍️ Toy 5 · SIFT-style descriptors rotate angles into a local frame

Subtracting the dominant orientation expresses all gradient directions relative to the keypoint rather
than the image axes.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)                 # seeded generator for this toy
t5_angles = np.array([20., 30., 200., 210., 95., 100.]) # -> absolute gradient angles
t5_mags = np.array([3., 2., 3., 2., 1., 1.])     # -> magnitude weights
print("angles:", t5_angles.tolist())             # -> [20.0, 30.0, 200.0, 210.0, 95.0, 100.0]
print("magnitudes:", t5_mags.tolist())           # -> [3.0, 2.0, 3.0, 2.0, 1.0, 1.0]
t5_edges = np.arange(0, 361, 45)                 # -> 45-degree bins
t5_hist_abs, _ = np.histogram(t5_angles, bins=t5_edges, weights=t5_mags) # -> [5,0,2,0,5,0,0,0]
print("absolute histogram:", t5_hist_abs.tolist()) # -> [5.0, 0.0, 2.0, 0.0, 5.0, 0.0, 0.0, 0.0]
t5_dom_bin = int(np.argmax(t5_hist_abs))         # -> 0
t5_dom_angle = 22.5 + 45 * t5_dom_bin            # -> 22.5
print("dominant angle:", t5_dom_angle)           # -> 22.5
t5_relative = (t5_angles - t5_dom_angle) % 360   # -> [357.5,7.5,177.5,187.5,72.5,77.5]
print("relative angles:", np.round(t5_relative, 1).tolist()) # -> [357.5, 7.5, 177.5, 187.5, 72.5, 77.5]
t5_hist_rel, _ = np.histogram(t5_relative, bins=t5_edges, weights=t5_mags) # -> [2,2,0,3,2,0,0,3]
print("relative histogram:", t5_hist_rel.tolist()) # -> [2.0, 2.0, 0.0, 3.0, 2.0, 0.0, 0.0, 3.0]
assert t5_dom_angle == 22.5 and t5_hist_rel.sum() == t5_mags.sum()

plt.figure(figsize=(5.2, 2.7))
plt.bar(np.arange(8) - 0.18, t5_hist_abs, width=0.36, label="absolute")
plt.bar(np.arange(8) + 0.18, t5_hist_rel, width=0.36, label="relative")
plt.xlabel("45° bin")
plt.ylabel("weighted votes")
plt.legend()
plt.title("Toy 5 · rotate descriptor frame")
plt.show()

▶ What you'll see: the same total gradient energy shifts bins after orientation normalization.

### ✍️ Toy 6 · Hough voting turns edge points into a line peak

Each edge point votes in parameter space. Several points on `x=2` agree at `theta=0°, rho=2`.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)                 # seeded generator for this toy
t6_points = np.array([[2., 0.], [2., 1.], [2., 3.], [0., 0.]]) # -> three line points plus one origin
t6_thetas = np.array([0., 45., 90.])             # -> candidate line angles
print("edge points:", t6_points.tolist())        # -> [[2.0, 0.0], [2.0, 1.0], [2.0, 3.0], [0.0, 0.0]]
t6_rhos = []                                     # -> rho rows by theta
for t6_theta in np.radians(t6_thetas):
    t6_rhos.append(t6_points[:, 0] * np.cos(t6_theta) + t6_points[:, 1] * np.sin(t6_theta))
t6_rhos = np.array(t6_rhos)                      # -> shape (3, 4)
print("rho votes:", np.round(t6_rhos, 3).tolist()) # -> [[2.0, 2.0, 2.0, 0.0], [1.414, 2.121, 3.536, 0.0], [0.0, 1.0, 3.0, 0.0]]
t6_bins = np.arange(0, 6)                        # -> rho bins 0..5
t6_acc = np.zeros((len(t6_thetas), len(t6_bins))) # -> accumulator grid
for t6_t in range(len(t6_thetas)):
    for t6_rho in t6_rhos[t6_t]:
        t6_nearest = int(np.argmin(np.abs(t6_bins - t6_rho)))
        t6_acc[t6_t, t6_nearest] += 1
print("accumulator:", t6_acc.astype(int).tolist()) # -> [[1, 0, 3, 0, 0, 0], [1, 1, 1, 0, 1, 0], [2, 1, 0, 1, 0, 0]]
t6_best = np.unravel_index(np.argmax(t6_acc), t6_acc.shape) # -> (0, 2)
print("best theta,rho:", float(t6_thetas[t6_best[0]]), int(t6_bins[t6_best[1]])) # -> 0.0 2
assert t6_acc[0, 2] == 3 and t6_best == (0, 2)

plt.figure(figsize=(5, 2.8))
plt.imshow(t6_acc, cmap="magma", aspect="auto")
plt.xticks(range(len(t6_bins)), t6_bins)
plt.yticks(range(len(t6_thetas)), t6_thetas)
plt.xlabel("rho bin")
plt.ylabel("theta")
plt.colorbar(label="votes")
plt.title("Toy 6 · Hough accumulator")
plt.show()

▶ What you'll see: one accumulator cell has three votes, revealing the vertical line.

### ✍️ Toy 7 · Pooling and nonmaximum suppression keep local winners

Pooling summarizes blocks, while nonmaximum suppression keeps peaks and drops nearby lower scores.

In [ ]:
import numpy as np

t7_rng = np.random.default_rng(0)                 # seeded generator for this toy
t7_act = np.array([[1., 4., 2., 0.],
                   [3., 6., 5., 1.],
                   [2., 1., 8., 7.],
                   [0., 2., 9., 3.]])             # -> activation map
print("activation map:", t7_act.tolist())        # -> [[1.0, 4.0, 2.0, 0.0], [3.0, 6.0, 5.0, 1.0], [2.0, 1.0, 8.0, 7.0], [0.0, 2.0, 9.0, 3.0]]
t7_pool = np.zeros((2, 2))                       # -> max-pooled map
for t7_i in range(2):
    for t7_j in range(2):
        t7_pool[t7_i, t7_j] = np.max(t7_act[2*t7_i:2*t7_i+2, 2*t7_j:2*t7_j+2])
print("max pool:", t7_pool.tolist())             # -> [[6.0, 5.0], [2.0, 9.0]]
t7_scores = np.array([0.2, 0.7, 0.6, 0.9, 0.85, 0.4]) # detector scores along a line
print("scores:", t7_scores.tolist())             # -> [0.2, 0.7, 0.6, 0.9, 0.85, 0.4]
t7_keep = np.zeros_like(t7_scores, dtype=bool)   # local maximum mask
for t7_idx in range(1, len(t7_scores) - 1):
    t7_keep[t7_idx] = t7_scores[t7_idx] >= t7_scores[t7_idx - 1] and t7_scores[t7_idx] >= t7_scores[t7_idx + 1]
print("kept peak indices:", np.where(t7_keep)[0].tolist()) # -> [1, 3]
assert np.array_equal(t7_pool, np.array([[6., 5.], [2., 9.]])) and np.array_equal(np.where(t7_keep)[0], [1, 3])

fig, t7_ax = plt.subplots(1, 2, figsize=(6.6, 2.5))
t7_ax[0].imshow(t7_pool, cmap="viridis")
t7_ax[0].set_title("max pooled")
t7_ax[1].plot(t7_scores, marker="o")
t7_ax[1].scatter(np.where(t7_keep)[0], t7_scores[t7_keep], color="red")
t7_ax[1].set_title("NMS peaks")
plt.suptitle("Toy 7 · local winners survive")
plt.show()

▶ What you'll see: pooling keeps block maxima, and NMS keeps only score peaks at indices `1` and `3`.

### ✍️ Toy 8 · Descriptor normalization and ratio tests compare shapes, not scale

Normalize descriptor vectors before measuring distance, then compare the nearest and second-nearest
matches with a ratio.

In [ ]:
import numpy as np

t8_rng = np.random.default_rng(0)                 # seeded generator for this toy
t8_query = np.array([3., 4., 0., 0.])             # -> unnormalized descriptor
print("query descriptor:", t8_query.tolist())    # -> [3.0, 4.0, 0.0, 0.0]
t8_query_norm = t8_query / np.linalg.norm(t8_query) # -> [0.6, 0.8, 0.0, 0.0]
print("normalized query:", t8_query_norm.tolist()) # -> [0.6, 0.8, 0.0, 0.0]
t8_candidates = np.array([[0.64, 0.77, 0., 0.],
                          [0., 1., 0., 0.],
                          [-0.6, 0.8, 0., 0.]]) # -> candidate descriptors
print("raw candidates:", t8_candidates.tolist()) # -> [[0.64, 0.77, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [-0.6, 0.8, 0.0, 0.0]]
t8_candidates_norm = t8_candidates / np.linalg.norm(t8_candidates, axis=1, keepdims=True) # row-normalized
print("normalized candidates:", np.round(t8_candidates_norm, 3).tolist()) # -> [[0.639, 0.769, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0], [-0.6, 0.8, 0.0, 0.0]]
t8_dist = np.linalg.norm(t8_candidates_norm - t8_query_norm, axis=1) # -> [0.04999, 0.63246, 1.2]
print("distances:", np.round(t8_dist, 3).tolist()) # -> [0.05, 0.632, 1.2]
t8_order = np.argsort(t8_dist)                    # -> [0, 1, 2]
print("nearest order:", t8_order.tolist())        # -> [0, 1, 2]
t8_ratio = float(t8_dist[t8_order[0]] / t8_dist[t8_order[1]]) # -> 0.07904
print("nearest/second ratio:", round(t8_ratio, 3)) # -> 0.079
assert t8_order[0] == 0 and t8_ratio < 0.1

plt.figure(figsize=(4.6, 2.6))
plt.bar(["cand 0", "cand 1", "cand 2"], t8_dist, color=["seagreen", "gray", "gray"])
plt.ylabel("L2 distance")
plt.title("Toy 8 · ratio test favors one clear match")
plt.show()

▶ What you'll see: candidate 0 is much closer than candidate 1, so the ratio is small.


## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for image arrays, gradients, histograms, votes, and linear algebra.
import matplotlib.pyplot as plt # load Matplotlib for heatmaps, bars, quivers, and diagnostic curves.
np.random.seed(0) # make all random examples reproducible across notebook runs.

def finite_gradients(I): # compute centered finite differences for a 2-D grayscale image.
    I = np.asarray(I, dtype=float) # ensure predictable floating-point math.
    gx = np.zeros_like(I) # allocate horizontal derivative image.
    gy = np.zeros_like(I) # allocate vertical derivative image.
    gx[:, 1:-1] = (I[:, 2:] - I[:, :-2]) / 2 # centered x difference.
    gy[1:-1, :] = (I[2:, :] - I[:-2, :]) / 2 # centered y difference.
    return gx, gy # return both derivative components.

def conv2_valid(I, K): # slide a small kernel over all valid windows.
    I = np.asarray(I, dtype=float); K = np.asarray(K, dtype=float) # convert inputs to arrays.
    out = np.zeros((I.shape[0] - K.shape[0] + 1, I.shape[1] - K.shape[1] + 1)) # output feature map.
    for i in range(out.shape[0]): # visit each valid row.
        for j in range(out.shape[1]): # visit each valid column.
            out[i, j] = np.sum(I[i:i+K.shape[0], j:j+K.shape[1]] * K) # window dot product.
    return out # return the feature map.

def show_img(M, title, cmap="viridis"): # define a compact image-display helper.
    plt.figure(figsize=(4, 3)) # create a small figure.
    plt.imshow(M, cmap=cmap, aspect="auto") # draw the matrix as an image.
    plt.colorbar(label="value") # add a numeric color scale.
    plt.title(title) # title the plot.
    plt.axis("off") # hide tick labels for image-like arrays.
    plt.show() # display the figure.

## 🟢 Basics (warm-up)

### Basic 1 — Make a tiny image patch

**Goal.** Store a grayscale patch as a NumPy array, because classical vision starts with numbers before it starts with features. We build it in 2 steps.

In [ ]:
patch_b1 = np.array([[1., 2.], [3., 4.]]) # create the exact 2x2 patch from the lesson mathematics.
print("patch shape:", patch_b1.shape) # inspect the image dimensions.
print(patch_b1) # inspect intensities before applying any feature.

▶ What you'll see: a 2×2 grid where the bottom-right pixel is largest.

In [ ]:
show_img(patch_b1, "Basic 1: grayscale patch", cmap="gray") # visualize the patch as an image.
assert patch_b1[1, 1] == 4.0 # verify the concrete patch value used by later filters.

▶ What you'll see: brighter color in the bottom-right, matching the larger intensity.

👀 Takeaway: images are arrays, and every classical feature is a measurement computed from those array values.

### Basic 2 — Apply one diagonal-difference filter

**Goal.** Compute one hand-made filter response, because a feature detector is a weighted comparison inside a local patch. We build it in 2 steps.

In [ ]:
patch_b2 = np.array([[1., 2.], [3., 4.]]) # recreate the local image patch for this example.
kernel_b2 = np.array([[1., 0.], [0., -1.]]) # define a diagonal comparison kernel.
products_b2 = patch_b2 * kernel_b2 # multiply pixel by pixel to reveal contributions.
print("products:\n", products_b2) # inspect which pixels actually contribute.

▶ What you'll see: the top-left contributes +1 and the bottom-right contributes −4.

In [ ]:
response_b2 = float(np.sum(products_b2)) # sum products into one filter response.
print("response:", response_b2) # inspect the signed feature value.
assert response_b2 == -3.0 # verify the lesson's diagonal-filter result.
plt.figure(figsize=(4, 3)); plt.bar(["+ corner", "- corner"], [1, -4], color=["teal", "crimson"]); plt.title("Basic 2: signed contributions"); plt.show()

▶ What you'll see: the negative contribution dominates, so the filter response is negative.

👀 Takeaway: a filter response is just a dot product between chosen weights and local pixels.

### Basic 3 — Slide the filter over a ramp

**Goal.** Turn one local filter into a feature map, because image features must be computed at many positions. We build it in 2 steps.

In [ ]:
ramp_b3 = np.arange(1, 10, dtype=float).reshape(3, 3) # create the 1-to-9 ramp from the lesson.
kernel_b3 = np.array([[1., 0.], [0., -1.]]) # use the same diagonal-difference filter.
feat_b3 = conv2_valid(ramp_b3, kernel_b3) # slide the filter over all valid 2x2 windows.
print("feature map:\n", feat_b3) # inspect all local responses.

▶ What you'll see: all four responses equal −4 because every diagonal gap in the ramp is the same.

In [ ]:
assert np.all(feat_b3 == -4.0) # verify the four canonical ramp responses.
fig, ax = plt.subplots(1, 2, figsize=(5.5, 2.6)); ax[0].imshow(ramp_b3); ax[0].set_title("ramp"); ax[1].imshow(feat_b3, vmin=-5, vmax=0); ax[1].set_title("responses"); plt.show()

▶ What you'll see: a 3×3 input becomes a 2×2 response map under a valid 2×2 filter.

👀 Takeaway: sliding-window feature maps report where the same local structure appears.

### Basic 4 — Compute horizontal and vertical gradients

**Goal.** Approximate local derivatives with finite differences, because edges are intensity changes rather than raw brightness values. We build it in 2 steps.

In [ ]:
img_b4 = np.array([[0., 0., 1., 1.], [0., 0., 1., 1.], [0., 0., 1., 1.], [0., 0., 1., 1.]]) # make a vertical step image.
Gx_b4, Gy_b4 = finite_gradients(img_b4) # compute centered x and y finite differences.
print("Gx:\n", Gx_b4) # inspect left-right changes.
print("Gy sum:", float(np.sum(np.abs(Gy_b4)))) # inspect top-bottom changes.

▶ What you'll see: `Gx` is nonzero around the vertical step while `Gy` is zero.

In [ ]:
assert np.sum(np.abs(Gy_b4)) == 0.0 # verify there is no vertical change in this toy image.
show_img(Gx_b4, "Basic 4: horizontal finite differences", cmap="coolwarm") # visualize the derivative image.

▶ What you'll see: two colored columns mark where intensity changes left-to-right.

👀 Takeaway: gradients detect changes, so they are less tied to absolute illumination than raw pixels.

### Basic 5 — Convert gradients to magnitude and angle

**Goal.** Combine `Gx` and `Gy` into edge strength and orientation, because descriptors need both how strong a change is and which way it points. We build it in 2 steps.

In [ ]:
Gx_b5 = np.array([[0., 3.], [0., -3.]]) # define horizontal gradient examples.
Gy_b5 = np.array([[4., 0.], [2., 4.]]) # define vertical gradient examples.
mag_b5 = np.sqrt(Gx_b5**2 + Gy_b5**2) # compute Euclidean gradient magnitude.
ang_b5 = np.degrees(np.arctan2(Gy_b5, Gx_b5)) # compute orientation in degrees.
print("magnitudes:\n", mag_b5) # inspect edge strengths.
print("angles:\n", np.round(ang_b5, 1)) # inspect edge directions.

▶ What you'll see: the (3,4) vector has magnitude 5 and angle about 53.1°.

In [ ]:
assert round(float(mag_b5[0, 1]), 3) == 3.0 # verify one concrete magnitude.
assert round(float(ang_b5[0, 0]), 1) == 90.0 # verify a pure vertical gradient angle.
show_img(mag_b5, "Basic 5: gradient magnitude", cmap="magma") # visualize edge strength.

▶ What you'll see: brighter cells are stronger gradients regardless of direction.

👀 Takeaway: magnitude and angle are the raw ingredients for edges, HOG, SIFT, and Hough voting.

### Basic 6 — Build a two-bin HOG cell

**Goal.** Pool several gradient vectors into an orientation histogram, because HOG describes a patch by its dominant local directions. We build it in 3 steps.

In [ ]:
grads_b6 = np.array([[3., 4.], [0., 2.], [-3., 4.], [4., 0.]]) # four gradient vectors in one cell.
mag_b6 = np.sqrt(np.sum(grads_b6**2, axis=1)) # compute each vector magnitude.
ang_b6 = (np.degrees(np.arctan2(grads_b6[:, 1], grads_b6[:, 0])) + 180) % 180 # unsigned angles.
print("angles:", np.round(ang_b6, 1)) # inspect which bins the gradients should enter.

▶ What you'll see: three vectors are upright-ish and one points horizontally.

In [ ]:
hist_b6 = np.zeros(2) # allocate two orientation bins: horizontal-ish and vertical-ish.
for m_b6, a_b6 in zip(mag_b6, ang_b6): # vote each gradient into a bin.
    hist_b6[0 if a_b6 < 45 or a_b6 >= 135 else 1] += m_b6 # weight by magnitude.
print("HOG histogram:", hist_b6) # inspect pooled orientation evidence.
assert np.allclose(hist_b6, [4., 12.]) # verify the lesson histogram numbers.

In [ ]:
plt.figure(figsize=(4, 3)); plt.bar(["horizontal", "vertical"], hist_b6, color="orange"); plt.title("Basic 6: HOG votes"); plt.ylabel("magnitude sum"); plt.show()

▶ What you'll see: the vertical-ish bin is three times larger than the horizontal-ish bin.

👀 Takeaway: HOG gains robustness by keeping orientation counts instead of exact pixel identities.

### Basic 7 — Normalize a descriptor vector

**Goal.** L2-normalize a histogram, because many classical descriptors care about relative pattern more than total contrast. We build it in 2 steps.

In [ ]:
hist_b7 = np.array([4., 12.]) # reuse the two-bin HOG histogram.
norm_b7 = np.linalg.norm(hist_b7) # compute descriptor length.
desc_b7 = hist_b7 / norm_b7 # normalize to unit length.
print("norm:", round(norm_b7, 3)) # inspect the scale being removed.
print("normalized descriptor:", np.round(desc_b7, 3)) # inspect contrast-normalized feature.

▶ What you'll see: the larger bin remains larger, but the descriptor length becomes 1.

In [ ]:
assert round(float(np.linalg.norm(desc_b7)), 6) == 1.0 # verify unit-length normalization.
plt.figure(figsize=(4, 3)); plt.bar(["bin0", "bin1"], desc_b7, color="seagreen"); plt.title("Basic 7: normalized HOG"); plt.show()

▶ What you'll see: normalization preserves the 1:3 ratio while changing the absolute scale.

👀 Takeaway: descriptor normalization buys contrast robustness but discards absolute edge strength.

### Basic 8 — Detect a simple corner from gradient energy

**Goal.** Compare horizontal and vertical gradient energy in a neighborhood, because corners have evidence in both directions. We build it in 3 steps.

In [ ]:
I_b8 = np.zeros((5, 5)); I_b8[2:, 2:] = 1.0 # create a small quadrant corner.
Gx_b8, Gy_b8 = finite_gradients(I_b8) # compute gradients around the corner.
win_b8 = (slice(1, 4), slice(1, 4)) # choose a local window around the corner.
Sxx_b8 = float(np.sum(Gx_b8[win_b8]**2)); Syy_b8 = float(np.sum(Gy_b8[win_b8]**2)) # summarize x/y energy.
print("Sxx, Syy:", Sxx_b8, Syy_b8) # inspect two-direction energy.

▶ What you'll see: both horizontal and vertical gradient energy are positive near the corner.

In [ ]:
Sxy_b8 = float(np.sum(Gx_b8[win_b8] * Gy_b8[win_b8])) # summarize mixed gradient energy.
M_b8 = np.array([[Sxx_b8, Sxy_b8], [Sxy_b8, Syy_b8]]) # assemble the second-moment matrix.
corner_b8 = np.linalg.det(M_b8) - 0.04 * np.trace(M_b8)**2 # compute a Harris-style response.
print("corner response:", round(float(corner_b8), 3)) # inspect whether the response is positive.
assert corner_b8 > 0 # verify this patch behaves corner-like.

In [ ]:
show_img(I_b8, "Basic 8: quadrant corner", cmap="gray") # visualize the corner input.

▶ What you'll see: an L-shaped transition, the simplest place where two edge directions meet.

👀 Takeaway: corners are valuable because their neighborhoods are localized in both x and y.

### Basic 9 — Vote for a vertical Hough line

**Goal.** Accumulate votes for line parameters, because many weak edge pixels can jointly reveal one geometric object. We build it in 2 steps.

In [ ]:
pts_b9 = np.array([[2., 0.], [2., 1.], [2., 3.]]) # three points on the same vertical line.
theta_b9 = 0.0 # test the vertical-line angle in degrees for rho = x.
rho_b9 = pts_b9[:, 0] * np.cos(np.radians(theta_b9)) + pts_b9[:, 1] * np.sin(np.radians(theta_b9)) # compute rho for each point.
print("rho votes:", rho_b9) # inspect votes for theta=0 degrees.
assert np.allclose(rho_b9, [2., 2., 2.]) # verify all points agree on rho=2.

▶ What you'll see: all three points vote for the same line bin.

In [ ]:
plt.figure(figsize=(4, 3)); plt.scatter(pts_b9[:, 0], pts_b9[:, 1], s=80); plt.axvline(2, color="red", linestyle="--"); plt.title("Basic 9: points vote for x=2"); plt.xlabel("x"); plt.ylabel("y"); plt.show()

▶ What you'll see: the red line passes through all three edge points.

👀 Takeaway: Hough voting converts local edge evidence into global shape evidence.

### Basic 10 — Max-pool a feature map

**Goal.** Keep the strongest response in each small block, because pooling makes a feature less sensitive to tiny shifts. We build it in 2 steps.

In [ ]:
act_b10 = np.array([[1., 3., 2., 0.], [4., 6., 5., 1.], [1., 2., 9., 8.], [0., 1., 7., 3.]]) # activation map to pool.
pooled_b10 = np.zeros((2, 2)) # allocate pooled output.
for i_b10 in range(2):
    for j_b10 in range(2):
        pooled_b10[i_b10, j_b10] = np.max(act_b10[2*i_b10:2*i_b10+2, 2*j_b10:2*j_b10+2]) # block maximum.
print("pooled:\n", pooled_b10) # inspect pooled values.

▶ What you'll see: each 2×2 block contributes one number.

In [ ]:
assert np.allclose(pooled_b10, [[6., 5.], [2., 9.]]) # verify the canonical pooling result.
fig, ax = plt.subplots(1, 2, figsize=(5.4, 2.6)); ax[0].imshow(act_b10); ax[0].set_title("activation"); ax[1].imshow(pooled_b10); ax[1].set_title("max pooled"); plt.show()

▶ What you'll see: the pooled map is smaller but retains the strongest local evidence.

👀 Takeaway: pooling buys local shift tolerance by discarding exact within-block positions.

## 🟡 Easy

### Easy 1 — Build a Sobel-style edge detector

**Goal.** Use larger hand-designed kernels for horizontal and vertical edges, because smoothing inside the kernel makes finite differences less brittle. We build it in 3 steps.

In [ ]:
img_e1 = np.zeros((7, 7)); img_e1[:, 3:] = 1.0 # create a clean vertical step edge.
Kx_e1 = np.array([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]) # Sobel-style x kernel.
Ky_e1 = Kx_e1.T # Sobel-style y kernel.
print("Kx sum:", Kx_e1.sum(), "Ky sum:", Ky_e1.sum()) # verify derivative kernels have zero DC response.
assert Kx_e1.sum() == 0.0

▶ What you'll see: the kernels sum to zero, so uniform brightness produces no edge response.

In [ ]:
Gx_e1 = conv2_valid(img_e1, Kx_e1) # compute horizontal edge responses.
Gy_e1 = conv2_valid(img_e1, Ky_e1) # compute vertical edge responses.
mag_e1 = np.sqrt(Gx_e1**2 + Gy_e1**2) # combine response components.
print("max magnitude:", float(np.max(mag_e1))) # inspect strongest edge score.
assert np.max(mag_e1) == 4.0

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6)); ax[0].imshow(img_e1, cmap="gray"); ax[0].set_title("image"); ax[1].imshow(Gx_e1, cmap="coolwarm"); ax[1].set_title("Gx"); ax[2].imshow(mag_e1, cmap="magma"); ax[2].set_title("magnitude"); plt.show()

▶ What you'll see: the Sobel response forms a strong vertical band at the step.

👀 Takeaway: edge filters are engineered finite differences with local smoothing built into the weights.

### Easy 2 — Compute HOG over several cells

**Goal.** Build a tiny HOG descriptor from a gradient field, because real HOG concatenates many local cell histograms. We build it in 3 steps.

In [ ]:
Gx_e2 = np.array([[1., 1., 0., 0.], [1., 1., 0., 0.], [0., 0., 2., 2.], [0., 0., 2., 2.]]) # horizontal gradients in two cells.
Gy_e2 = np.array([[0., 0., 2., 2.], [0., 0., 2., 2.], [1., 1., 0., 0.], [1., 1., 0., 0.]]) # vertical gradients in other cells.
mag_e2 = np.sqrt(Gx_e2**2 + Gy_e2**2) # gradient strength per pixel.
ang_e2 = (np.degrees(np.arctan2(Gy_e2, Gx_e2)) + 180) % 180 # unsigned orientations.
print("total magnitude:", round(float(mag_e2.sum()), 3)) # inspect descriptor evidence scale.

▶ What you'll see: all pixels contribute positive gradient evidence.

In [ ]:
desc_e2 = [] # collect one two-bin histogram per 2x2 cell.
for i_e2 in [0, 2]:
    for j_e2 in [0, 2]:
        h_e2 = np.zeros(2) # horizontal-ish and vertical-ish bins.
        for m_e2, a_e2 in zip(mag_e2[i_e2:i_e2+2, j_e2:j_e2+2].ravel(), ang_e2[i_e2:i_e2+2, j_e2:j_e2+2].ravel()):
            h_e2[0 if a_e2 < 45 or a_e2 >= 135 else 1] += m_e2 # magnitude-weighted vote.
        desc_e2.extend(h_e2) # concatenate cell histogram.
desc_e2 = np.array(desc_e2) # convert to descriptor array.
print("HOG descriptor:", desc_e2) # inspect concatenated cell features.
assert desc_e2.shape == (8,)

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(np.arange(len(desc_e2)), desc_e2, color="purple"); plt.title("Easy 2: four-cell HOG descriptor"); plt.xlabel("descriptor coordinate"); plt.show()

▶ What you'll see: different cells prefer different orientation bins, so location is coarse but not fully lost.

👀 Takeaway: HOG is a concatenation of local orientation summaries, not one global edge count.

### Easy 3 — Make a simple SIFT-like patch descriptor

**Goal.** Normalize a patch descriptor by dominant orientation and length, because SIFT aims to match repeated local structure under rotation and contrast changes. We build it in 4 steps.

In [ ]:
angles_e3 = np.array([20., 35., 40., 200., 215., 220., 90., 95.]) # toy keypoint-neighborhood angles.
mags_e3 = np.array([3., 2., 4., 3., 2., 4., 1., 1.]) # magnitudes for weighted votes.
bins_e3 = np.arange(0, 361, 45) # eight orientation bins.
hist_e3, _ = np.histogram(angles_e3, bins=bins_e3, weights=mags_e3) # absolute orientation histogram.
print("absolute hist:", hist_e3) # inspect dominant orientation evidence.

▶ What you'll see: the first orientation bin receives the largest weighted vote.

In [ ]:
dom_e3 = 22.5 + 45 * int(np.argmax(hist_e3)) # use the center of the winning bin.
rel_e3 = (angles_e3 - dom_e3) % 360 # rotate all angles into the keypoint coordinate frame.
rel_hist_e3, _ = np.histogram(rel_e3, bins=bins_e3, weights=mags_e3) # histogram relative orientations.
print("dominant angle:", dom_e3) # inspect the chosen local frame.
print("relative hist:", rel_hist_e3) # inspect orientation-normalized descriptor.
assert rel_hist_e3.sum() == mags_e3.sum()

In [ ]:
desc_e3 = rel_hist_e3 / (np.linalg.norm(rel_hist_e3) + 1e-12) # length-normalize the descriptor.
print("descriptor norm:", round(float(np.linalg.norm(desc_e3)), 6)) # verify contrast normalization.
assert round(float(np.linalg.norm(desc_e3)), 6) == 1.0

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(np.arange(8), desc_e3, color="seagreen"); plt.title("Easy 3: normalized SIFT-like descriptor"); plt.xlabel("relative orientation bin"); plt.show()

▶ What you'll see: the descriptor is expressed in relative orientation bins and has unit length.

👀 Takeaway: SIFT-style normalization intentionally loses absolute orientation and contrast to gain match stability.

### Easy 4 — Score corners across a tiny image

**Goal.** Compute a Harris-style response map, because corner detectors must find keypoints rather than just score one patch. We build it in 4 steps.

In [ ]:
I_e4 = np.zeros((8, 8)); I_e4[3:, 3:] = 1.0 # create an L corner in a small image.
Gx_e4, Gy_e4 = finite_gradients(I_e4) # compute finite-difference gradients.
R_e4 = np.zeros_like(I_e4) # allocate corner response map.
print("image shape:", I_e4.shape) # inspect the grid being scored.

▶ What you'll see: a small image with one quadrant transition.

In [ ]:
for i_e4 in range(1, 7):
    for j_e4 in range(1, 7):
        w_e4 = (slice(i_e4-1, i_e4+2), slice(j_e4-1, j_e4+2)) # 3x3 local window.
        Sxx_e4 = np.sum(Gx_e4[w_e4]**2); Syy_e4 = np.sum(Gy_e4[w_e4]**2); Sxy_e4 = np.sum(Gx_e4[w_e4] * Gy_e4[w_e4]) # local products.
        M_e4 = np.array([[Sxx_e4, Sxy_e4], [Sxy_e4, Syy_e4]]) # second-moment matrix.
        R_e4[i_e4, j_e4] = np.linalg.det(M_e4) - 0.04 * np.trace(M_e4)**2 # Harris-style score.
print("best corner index:", np.unravel_index(np.argmax(R_e4), R_e4.shape)) # inspect strongest response location.

In [ ]:
best_e4 = np.unravel_index(np.argmax(R_e4), R_e4.shape) # read the peak coordinate.
assert best_e4 in [(2, 2), (2, 3), (3, 2), (3, 3)] # verify the peak lies around the true corner.
print("max response:", round(float(np.max(R_e4)), 3)) # inspect peak strength.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(5.5, 2.6)); ax[0].imshow(I_e4, cmap="gray"); ax[0].set_title("input"); ax[1].imshow(R_e4, cmap="magma"); ax[1].scatter([best_e4[1]], [best_e4[0]], c="cyan"); ax[1].set_title("corner response"); plt.show()

▶ What you'll see: the brightest response appears near the L-shaped junction.

👀 Takeaway: corner scoring searches for neighborhoods with strong gradient variation in two directions.

### Easy 5 — Build a Hough accumulator image

**Goal.** Vote over several line angles and rho bins, because the Hough transform detects geometric agreement as peaks in parameter space. We build it in 4 steps.

In [ ]:
pts_e5 = np.array([[2., 0.], [2., 1.], [2., 3.], [0., 4.]]) # three line points plus one off-line point.
thetas_e5 = np.array([0., 45., 90.]) # small angle grid for a readable accumulator.
rho_bins_e5 = np.arange(0, 7) # integer rho bins.
acc_e5 = np.zeros((len(thetas_e5), len(rho_bins_e5))) # allocate accumulator.
print("points:", pts_e5.tolist()) # inspect voters.

▶ What you'll see: most points share x=2, with one distractor.

In [ ]:
for t_idx_e5, theta_e5 in enumerate(np.radians(thetas_e5)):
    for x_e5, y_e5 in pts_e5:
        rho_e5 = x_e5 * np.cos(theta_e5) + y_e5 * np.sin(theta_e5) # line parameter for this point and angle.
        b_e5 = int(np.argmin(np.abs(rho_bins_e5 - rho_e5))) # nearest rho bin.
        acc_e5[t_idx_e5, b_e5] += 1 # cast one vote.
print("accumulator:\n", acc_e5.astype(int)) # inspect vote counts.

In [ ]:
peak_e5 = np.unravel_index(np.argmax(acc_e5), acc_e5.shape) # find the strongest parameter bin.
print("peak theta/rho:", thetas_e5[peak_e5[0]], rho_bins_e5[peak_e5[1]]) # inspect detected line.
assert acc_e5[0, 2] == 3 # verify the vertical line got three votes.

In [ ]:
plt.figure(figsize=(5, 3)); plt.imshow(acc_e5, cmap="magma", aspect="auto"); plt.xticks(range(len(rho_bins_e5)), rho_bins_e5); plt.yticks(range(len(thetas_e5)), thetas_e5); plt.xlabel("rho"); plt.ylabel("theta"); plt.title("Easy 5: Hough votes"); plt.colorbar(label="votes"); plt.show()

▶ What you'll see: the brightest accumulator bin marks the shared vertical line.

👀 Takeaway: Hough detection is consensus finding in a parameter grid.

## 🔴 Advanced

### Advanced 1 — Thin edges with nonmaximum suppression

**Goal.** Keep only local maxima along a 1-D edge-strength profile, because raw gradients often make thick edge bands. We build it in 4 steps.

In [ ]:
profile_a1 = np.array([0., 0.2, 0.7, 1.0, 0.65, 0.1, 0., 0.5, 0.9, 0.4]) # toy gradient magnitudes along one row.
keep_a1 = np.zeros_like(profile_a1, dtype=bool) # allocate keep mask.
print("profile:", profile_a1) # inspect raw edge strengths.

▶ What you'll see: two broad hills of edge strength rather than two single-pixel edges.

In [ ]:
for i_a1 in range(1, len(profile_a1) - 1):
    keep_a1[i_a1] = profile_a1[i_a1] >= profile_a1[i_a1-1] and profile_a1[i_a1] >= profile_a1[i_a1+1] # local maximum test.
thin_a1 = np.where(keep_a1, profile_a1, 0.0) # suppress non-maximum responses.
print("kept indices:", np.where(keep_a1)[0]) # inspect thinned edge positions.
assert np.array_equal(np.where(keep_a1)[0], [3, 8])

In [ ]:
print("thinned profile:", thin_a1) # inspect remaining responses.
assert round(float(thin_a1.sum()), 3) == 1.9 # verify only the two peaks remain.

In [ ]:
plt.figure(figsize=(5, 3)); plt.plot(profile_a1, marker="o", label="raw magnitude"); plt.stem(np.arange(len(thin_a1)), thin_a1, linefmt="r-", markerfmt="ro", basefmt=" ", label="kept"); plt.title("Advanced 1: nonmaximum suppression"); plt.legend(); plt.show()

▶ What you'll see: each broad hill collapses to its strongest sample.

👀 Takeaway: nonmaximum suppression turns thick detections into localized peaks before thresholding or matching.

### Advanced 2 — Compare template matching with gradient matching

**Goal.** Show why gradients are often more stable than raw pixels under brightness changes, because classical features were designed to ignore nuisance variation. We build it in 4 steps.

In [ ]:
template_a2 = np.array([[0., 0., 1.], [0., 0., 1.], [0., 0., 1.]]) # vertical edge template.
bright_a2 = template_a2 + 5.0 # same edge under a large brightness offset.
raw_dist_a2 = float(np.linalg.norm(template_a2 - bright_a2)) # raw-pixel distance.
print("raw distance after brightness shift:", round(raw_dist_a2, 3)) # inspect brittleness.
assert round(raw_dist_a2, 3) == 15.0

▶ What you'll see: a pure brightness offset creates a large raw-pixel distance.

In [ ]:
Gxt_a2, Gyt_a2 = finite_gradients(template_a2) # gradients of original patch.
Gxb_a2, Gyb_a2 = finite_gradients(bright_a2) # gradients after brightness offset.
grad_dist_a2 = float(np.linalg.norm(Gxt_a2 - Gxb_a2) + np.linalg.norm(Gyt_a2 - Gyb_a2)) # compare derivative features.
print("gradient distance after brightness shift:", round(grad_dist_a2, 3)) # inspect invariance.
assert grad_dist_a2 == 0.0

In [ ]:
raw_sim_a2 = -raw_dist_a2 # larger is better as negative distance.
grad_sim_a2 = -grad_dist_a2 # larger is better as negative distance.
print("raw similarity:", raw_sim_a2, "gradient similarity:", grad_sim_a2) # compare scores.

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["raw pixels", "gradients"], [raw_dist_a2, grad_dist_a2], color=["crimson", "seagreen"]); plt.title("Advanced 2: brightness shift distance"); plt.ylabel("distance lower is better"); plt.show()

▶ What you'll see: raw pixels change dramatically, while gradients stay identical.

👀 Takeaway: finite differences deliberately discard absolute brightness so structure can match across lighting shifts.

### Advanced 3 — Sweep HOG cell size

**Goal.** Compare small and large HOG cells, because cell size controls the tradeoff between localization and shift robustness. We build it in 4 steps.

In [ ]:
Gx_a3 = np.zeros((8, 8)); Gy_a3 = np.zeros((8, 8)) # allocate a toy gradient field.
Gx_a3[:, 2] = 3.0; Gy_a3[5, :] = 2.0 # create a vertical edge and a horizontal edge.
cell_sizes_a3 = np.array([2, 4]) # compare fine and coarse cells.
print("nonzero gradient pixels:", int(np.sum((Gx_a3 != 0) | (Gy_a3 != 0)))) # inspect edge evidence count.

▶ What you'll see: the field contains two simple edge structures.

In [ ]:
lengths_a3 = [] # descriptor lengths for each cell size.
nonzero_bins_a3 = [] # active bins for each descriptor.
for c_a3 in cell_sizes_a3:
    desc_a3 = [] # one descriptor per cell size.
    for i_a3 in range(0, 8, c_a3):
        for j_a3 in range(0, 8, c_a3):
            mag_a3 = np.sqrt(Gx_a3[i_a3:i_a3+c_a3, j_a3:j_a3+c_a3]**2 + Gy_a3[i_a3:i_a3+c_a3, j_a3:j_a3+c_a3]**2) # local magnitude.
            ang_a3 = (np.degrees(np.arctan2(Gy_a3[i_a3:i_a3+c_a3, j_a3:j_a3+c_a3], Gx_a3[i_a3:i_a3+c_a3, j_a3:j_a3+c_a3])) + 180) % 180 # local angle.
            h_a3 = np.array([mag_a3[(ang_a3 < 45) | (ang_a3 >= 135)].sum(), mag_a3[(ang_a3 >= 45) & (ang_a3 < 135)].sum()]) # two-bin cell hist.
            desc_a3.extend(h_a3) # concatenate.
    desc_a3 = np.array(desc_a3) # convert descriptor.
    lengths_a3.append(len(desc_a3)); nonzero_bins_a3.append(int(np.sum(desc_a3 > 0))) # summarize.
print("descriptor lengths:", lengths_a3, "nonzero bins:", nonzero_bins_a3) # inspect tradeoff.

In [ ]:
assert lengths_a3 == [32, 8] # verify smaller cells produce longer descriptors.
assert nonzero_bins_a3[0] >= nonzero_bins_a3[1] # fine cells preserve more spatial detail.
print("fine/coarse length ratio:", lengths_a3[0] // lengths_a3[1]) # inspect compression.

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["2x2 cells", "4x4 cells"], lengths_a3, color="slateblue"); plt.title("Advanced 3: HOG descriptor length vs cell size"); plt.ylabel("descriptor coordinates"); plt.show()

▶ What you'll see: smaller cells produce a longer descriptor that preserves more approximate location.

👀 Takeaway: HOG cell size is an invariance knob: coarse cells are steadier, fine cells are more precise.

### Advanced 4 — Match two SIFT-like descriptors with ratio test

**Goal.** Use nearest-neighbor matching plus a ratio test, because SIFT matching accepts a feature only when its best match is clearly better than the runner-up. We build it in 4 steps.

In [ ]:
query_a4 = np.array([0.8, 0.2, 0.0, 0.1]) # one normalized-looking descriptor.
cands_a4 = np.array([[0.77, 0.22, 0.0, 0.09], [0.45, 0.55, 0.1, 0.0], [0.78, 0.18, 0.0, 0.12]]) # candidate descriptors.
dists_a4 = np.linalg.norm(cands_a4 - query_a4, axis=1) # Euclidean descriptor distances.
print("distances:", np.round(dists_a4, 3)) # inspect nearest candidates.

▶ What you'll see: candidate 2 is nearest, but candidate 0 is also close.

In [ ]:
order_a4 = np.argsort(dists_a4) # sort matches from closest to farthest.
best_a4, second_a4 = dists_a4[order_a4[0]], dists_a4[order_a4[1]] # read top two distances.
ratio_a4 = best_a4 / second_a4 # Lowe-style ambiguity ratio.
print("best index:", int(order_a4[0]), "ratio:", round(float(ratio_a4), 3)) # inspect confidence.

In [ ]:
accept_a4 = ratio_a4 < 0.8 # accept only if nearest is much better than second-nearest.
print("accepted match?", bool(accept_a4)) # inspect the decision.
assert accept_a4 == False # this toy match is ambiguous because two candidates are too close.

In [ ]:
plt.figure(figsize=(5, 3)); plt.bar(["best", "second", "third"], dists_a4[order_a4], color=["seagreen", "orange", "gray"]); plt.axhline(0.8 * second_a4, color="red", linestyle="--", label="0.8 × second"); plt.title("Advanced 4: SIFT ratio test"); plt.ylabel("distance"); plt.legend(); plt.show()

▶ What you'll see: the best distance is not below the red ambiguity threshold, so the match is rejected.

👀 Takeaway: descriptor matching needs an ambiguity check, not just the smallest distance.

### Advanced 5 — Combine features into a tiny classical detector

**Goal.** Build a miniature detection pipeline from gradients, thresholding, Hough voting, and pooling, because classical vision composes simple hand-made stages. We build it in 5 steps.

In [ ]:
I_a5 = np.zeros((10, 10)); I_a5[:, 5:] = 1.0; I_a5[7, 1:9] = 1.0 # create a vertical step plus a horizontal bright stroke.
Gx_a5, Gy_a5 = finite_gradients(I_a5) # compute image gradients.
mag_a5 = np.sqrt(Gx_a5**2 + Gy_a5**2) # edge strength.
edges_a5 = mag_a5 > 0.25 # threshold strong gradients.
print("edge pixel count:", int(edges_a5.sum())) # inspect detected edge evidence.
assert edges_a5.sum() > 0

▶ What you'll see: the threshold keeps pixels around the strong structures.

In [ ]:
ys_a5, xs_a5 = np.where(edges_a5) # convert edge mask to point coordinates.
thetas_a5 = np.array([0., 90.]) # vote only for vertical and horizontal lines for this tiny demo.
rho_bins_a5 = np.arange(0, 10) # integer rho bins.
acc_a5 = np.zeros((len(thetas_a5), len(rho_bins_a5))) # Hough accumulator.
for t_i_a5, th_a5 in enumerate(np.radians(thetas_a5)):
    for x_a5, y_a5 in zip(xs_a5, ys_a5):
        rho_a5 = x_a5 * np.cos(th_a5) + y_a5 * np.sin(th_a5) # line parameter.
        acc_a5[t_i_a5, int(np.argmin(np.abs(rho_bins_a5 - rho_a5)))] += 1 # nearest-bin vote.
print("best votes:", int(np.max(acc_a5))) # inspect strongest geometric consensus.

In [ ]:
peak_a5 = np.unravel_index(np.argmax(acc_a5), acc_a5.shape) # find the strongest line parameter.
print("detected theta/rho:", thetas_a5[peak_a5[0]], rho_bins_a5[peak_a5[1]]) # inspect line detection.
assert np.max(acc_a5) >= 8 # verify many edge pixels agreed on one line.

In [ ]:
pooled_a5 = np.zeros((5, 5)) # allocate a 2x2 max-pooled edge map.
for i_a5 in range(5):
    for j_a5 in range(5):
        pooled_a5[i_a5, j_a5] = np.max(mag_a5[2*i_a5:2*i_a5+2, 2*j_a5:2*j_a5+2]) # local strongest edge.
print("pooled edge max:", round(float(pooled_a5.max()), 3)) # inspect pooled evidence.
assert pooled_a5.max() > 0.0

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(8, 2.6)); ax[0].imshow(I_a5, cmap="gray"); ax[0].set_title("image"); ax[1].imshow(edges_a5, cmap="gray"); ax[1].set_title("edge mask"); ax[2].imshow(acc_a5, cmap="magma", aspect="auto"); ax[2].set_title("Hough votes"); plt.show()

▶ What you'll see: gradients create an edge mask, votes reveal a dominant line, and pooling would pass along stable local edge strength.

👀 Takeaway: a classical detector is a pipeline of explicit invariances and hand-designed measurements.